# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anasYaha/Flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Two signals my rule leans on**

Signal A — volume (flag-linked: behind FlyRank's `is_quick_win` flag). Claim: "higher-volume
queries attract more competing content items" — if true, volume is a fair way to prioritize
which cannibalized queries to fix first.

Signal B — fragmentation (`competing_content_count`). Claim: "as more of a client's own pages
compete for a query, any single page's share of that query's impressions shrinks."

Both are built only from honest features (never `top_content_impression_share`, which the
`is_cannibalized` proxy is thresholded on).

**The rule, in plain words:** a (client, query) pair is worth reviewing first if real search
volume is going to it AND multiple of the client's own pages are already competing for it.

- Score = percentile(log1p(query_total_impressions)) × percentile(competing_content_count)
- Reason code (one per row): high_volume_fragmented / moderate_fragmented / low_volume_fragmented
- Action: consolidate_or_canonicalize / review_for_consolidation / monitor

In [9]:
# Environment + table setup (per flyrank-data skill: HF_TOKEN via Colab Secret, never pasted)
%pip -q install duckdb huggingface_hub

import os, getpass, duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_mar': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
QUERY_ID_COL = 'query_hash_id'  # <-- same column confirmed in w03_data_contract.ipynb


In [10]:
# Same (client, query, content) feature frame as w03_data_contract.ipynb, restricted to
# content active in the mid-panel month=2026-03 slice (never the sealed final month).
cannib = con.sql(f"""
    WITH per_query AS (
        SELECT client_hash_id, {QUERY_ID_COL} AS query_id, content_hash_id,
               SUM(impressions_90d) AS content_impressions_for_query
        FROM {TABLES['fact_query_90d']}
        WHERE content_hash_id IN (SELECT DISTINCT content_hash_id FROM {TABLES['fact_daily_mar']})
        GROUP BY 1, 2, 3
    ),
    query_totals AS (
        SELECT client_hash_id, query_id,
               COUNT(DISTINCT content_hash_id)    AS competing_content_count,
               SUM(content_impressions_for_query)  AS query_total_impressions,
               MAX(content_impressions_for_query)  AS top_content_impressions
        FROM per_query
        GROUP BY 1, 2
    )
    SELECT p.client_hash_id, p.query_id, p.content_hash_id,
           p.content_impressions_for_query / NULLIF(qt.query_total_impressions, 0) AS own_impression_share_of_query,
           qt.competing_content_count,
           qt.query_total_impressions,
           qt.top_content_impressions / NULLIF(qt.query_total_impressions, 0)      AS top_content_impression_share
    FROM per_query p
    JOIN query_totals qt USING (client_hash_id, query_id)
""").df()

# Collapse to the query-level unit of analysis (one row = one (client, query) pair)
query_level = (
    cannib.groupby(['client_hash_id', 'query_id'], as_index=False)
    .agg(competing_content_count=('competing_content_count', 'first'),
         query_total_impressions=('query_total_impressions', 'first'),
         top_content_impression_share=('top_content_impression_share', 'first'))
)
print(f'{len(cannib):,} (client, query, content) rows | {len(query_level):,} (client, query) pairs')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2,128,659 (client, query, content) rows | 1,129,695 (client, query) pairs


In [11]:
# --- Signal A: volume vs. competing_content_count (flag-linked: behind is_quick_win) ---
volume_bins = [0, 100, 500, np.inf]
volume_labels = ['low (<100)', 'mid (100-499)', 'high (500+)']
query_level['volume_tier'] = pd.cut(query_level['query_total_impressions'],
                                     bins=volume_bins, labels=volume_labels, right=False)

signal_a = query_level.groupby('volume_tier', observed=True).agg(
    n=('competing_content_count', 'size'),
    mean_competing_content_count=('competing_content_count', 'mean'),
).reset_index()
print('Signal A — volume tier vs. mean competing_content_count (min bucket size: 50)')
print(signal_a)

# Verdict is written by hand in the markdown cell below AFTER reading this table —
# do not hardcode a verdict string here; read the printed n's and means first.


Signal A — volume tier vs. mean competing_content_count (min bucket size: 50)
     volume_tier       n  mean_competing_content_count
0     low (<100)  837724                      1.312605
1  mid (100-499)  228735                      2.795165
2    high (500+)   63236                      6.162724


**Verdict — Signal A (volume): _[fill in: CONFIRMED / OPPOSITE / MIXED / FALSE]_**

*Read the table above. If any tier has n < 50, say "insufficient data" for that tier instead of
a verdict on it — don't average a huge ratio out of a tiny cell. Write one sentence on what this
means in practice for the rule (e.g. "confirms volume is a safe first factor: higher-volume
queries really do attract more of the client's own competing pages, so ranking by volume first
is not just chasing noise").*


In [12]:
# --- Signal B: competing_content_count vs. own_impression_share_of_query ---
count_bins = [0, 1, 2, 4, np.inf]
count_labels = ['1 (no overlap)', '2', '3-4', '5+']
cannib['competitor_tier'] = pd.cut(cannib['competing_content_count'],
                                    bins=count_bins, labels=count_labels, right=True)

signal_b = cannib.groupby('competitor_tier', observed=True).agg(
    n=('own_impression_share_of_query', 'size'),
    mean_own_impression_share_of_query=('own_impression_share_of_query', 'mean'),
).reset_index()
print('Signal B — competitor tier vs. mean own_impression_share_of_query (min bucket size: 50)')
print(signal_b)


Signal B — competitor tier vs. mean own_impression_share_of_query (min bucket size: 50)
  competitor_tier       n  mean_own_impression_share_of_query
0  1 (no overlap)  751492                            1.000000
1               2  376524                            0.500000
2             3-4  391217                            0.299900
3              5+  609426                            0.119153


**Verdict — Signal B (fragmentation): _[fill in: CONFIRMED / OPPOSITE / MIXED / FALSE]_**

*Same rule: no verdict from a bucket under ~50 rows. One sentence on what it means for the rule
(e.g. "confirms competing_content_count is a fair fragmentation proxy: a page's own share of a
query's impressions really does shrink as more of the client's own pages compete for it").*


## 2. Build the ranked queue (writes the CSV)

Score only pairs with 2+ competing content items — a single-content "query" isn't cannibalized,
it's just a page. `top_content_impression_share` is carried into the output **for human context
only** — it is never multiplied into the score (it's the exact column `is_cannibalized` is
thresholded on in w03, so it's a label input, not a feature).


In [13]:
def percentile_rank(s: pd.Series) -> pd.Series:
    return s.rank(pct=True, method='average')

def reason_code(row) -> str:
    if row['volume_score'] >= 0.66 and row['fragmentation_score'] >= 0.66:
        return 'high_volume_fragmented'
    if row['volume_score'] >= 0.33 or row['fragmentation_score'] >= 0.66:
        return 'moderate_fragmented'
    return 'low_volume_fragmented'

def action_label(reason: str) -> str:
    return {
        'high_volume_fragmented': 'consolidate_or_canonicalize',
        'moderate_fragmented': 'review_for_consolidation',
        'low_volume_fragmented': 'monitor',
    }[reason]

candidates = query_level[query_level['competing_content_count'] >= 2].copy()

candidates['volume_score'] = percentile_rank(np.log1p(candidates['query_total_impressions']))
candidates['fragmentation_score'] = percentile_rank(candidates['competing_content_count'])
candidates['baseline_action_score'] = (candidates['volume_score'] * candidates['fragmentation_score']).round(4)

candidates['reason_code'] = candidates.apply(reason_code, axis=1)
candidates['action_label'] = candidates['reason_code'].apply(action_label)
candidates['rank'] = candidates['baseline_action_score'].rank(method='first', ascending=False).astype(int)

out_cols = ['rank', 'client_hash_id', 'query_id', 'baseline_action_score', 'reason_code',
            'action_label', 'competing_content_count', 'query_total_impressions',
            'top_content_impression_share']  # last column: context only, not a score input
queue = candidates[out_cols].sort_values('rank')

import pathlib
out_path = pathlib.Path('work/outputs/baseline_action_score.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(out_path, index=False)

print(f'Wrote {len(queue):,} ranked (client, query) pairs to {out_path}')
queue.head(10)


Wrote 378,203 ranked (client, query) pairs to work/outputs/baseline_action_score.csv


,rank,client_hash_id,query_id,baseline_action_score,reason_code,action_label,competing_content_count,query_total_impressions,top_content_impression_share
787476,1,client_73cda7b4e4f265ea,query_d741bc9c5f71aee4,1.0000,high_volume_fragmented,consolidate_or_canonicalize,839,247484.0,0.043045
537244,2,client_73cda7b4e4f265ea,query_05118e04631b9763,0.9999,high_volume_fragmented,consolidate_or_canonicalize,398,104849.0,0.057874
804762,3,client_73cda7b4e4f265ea,query_e5c1bca0d6d5fdf0,0.9999,high_volume_fragmented,consolidate_or_canonicalize,268,124214.0,0.359042
327503,4,client_23a62021009f63c4,query_e4c98f72e732501d,0.9998,high_volume_fragmented,consolidate_or_canonicalize,252,63814.0,0.104460
715232,5,client_73cda7b4e4f265ea,query_9a77450699268f30,0.9998,high_volume_fragmented,consolidate_or_canonicalize,471,58977.0,0.061244
735390,6,client_73cda7b4e4f265ea,query_ab81171134a428bb,0.9998,high_volume_fragmented,consolidate_or_canonicalize,113,292405.0,0.893251
762796,7,client_73cda7b4e4f265ea,query_c26522db2bbdf9a0,0.9998,high_volume_fragmented,consolidate_or_canonicalize,361,65123.0,0.073139
333993,8,client_23a62021009f63c4,query_ed55d56301b7ea3d,0.9997,high_volume_fragmented,consolidate_or_canonicalize,114,60639.0,0.733769
585710,9,client_73cda7b4e4f265ea,query_2d90b036c89fdaa0,0.9997,high_volume_fragmented,consolidate_or_canonicalize,130,70840.0,0.480759
191710,10,client_23a62021009f63c4,query_335a41dfb8b084d3,0.9996,high_volume_fragmented,consolidate_or_canonicalize,104,55628.0,0.589397


texte en italique## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

In [14]:
top10 = queue.head(10).reset_index(drop=True)
for i, row in top10.iterrows():
    print(f"{i+1}. action={row['action_label']} | reason={row['reason_code']} | "
          f"score={row['baseline_action_score']:.3f} | "
          f"{int(row['competing_content_count'])} competing pages, "
          f"{int(row['query_total_impressions']):,} query impressions, "
          f"top page currently holds {row['top_content_impression_share']*100:.0f}% of them (context only)\n"
          f"   Why it's here: high on both volume and fragmentation, so real impression volume\n"
          f"   looks split across the client's own pages instead of consolidated.\n"
          f"   What would make it wrong: if the competing pages actually serve different search\n"
          f"   intents (e.g. a buying guide vs. a product page) rather than duplicating one\n"
          f"   intent, consolidating would destroy legitimate distinct-intent coverage — this\n"
          f"   needs a human check of what the pages actually are before merging anything.\n")


1. action=consolidate_or_canonicalize | reason=high_volume_fragmented | score=1.000 | 839 competing pages, 247,484 query impressions, top page currently holds 4% of them (context only)
   Why it's here: high on both volume and fragmentation, so real impression volume
   looks split across the client's own pages instead of consolidated.
   What would make it wrong: if the competing pages actually serve different search
   intents (e.g. a buying guide vs. a product page) rather than duplicating one
   intent, consolidating would destroy legitimate distinct-intent coverage — this
   needs a human check of what the pages actually are before merging anything.

2. action=consolidate_or_canonicalize | reason=high_volume_fragmented | score=1.000 | 398 competing pages, 104,849 query impressions, top page currently holds 6% of them (context only)
   Why it's here: high on both volume and fragmentation, so real impression volume
   looks split across the client's own pages instead of consolidated

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [15]:
# --- Weak-pick check: flag rows where the score looks confident but the context column
# (top_content_impression_share, never used in the score) suggests it may already be fine ---
weak = top10[top10['top_content_impression_share'] >= 0.70]
print(f'{len(weak)} of the top 10 already have one page holding 70%+ of the query\'s impressions')
print('-> these are weak picks: high volume + several competing rows can still score well even')
print('   when one page already dominates, because the score never looks at the context column.')
weak


2 of the top 10 already have one page holding 70%+ of the query's impressions
-> these are weak picks: high volume + several competing rows can still score well even
   when one page already dominates, because the score never looks at the context column.


,rank,client_hash_id,query_id,baseline_action_score,reason_code,action_label,competing_content_count,query_total_impressions,top_content_impression_share
5,6,client_73cda7b4e4f265ea,query_ab81171134a428bb,0.9998,high_volume_fragmented,consolidate_or_canonicalize,113,292405.0,0.893251
7,8,client_23a62021009f63c4,query_ed55d56301b7ea3d,0.9997,high_volume_fragmented,consolidate_or_canonicalize,114,60639.0,0.733769


In [16]:
# --- Leakage check: confirm the score formula never touched a label-derived or future column ---
score_inputs = {'query_total_impressions', 'competing_content_count'}
forbidden = {'top_content_impression_share', 'is_cannibalized'}
print('Score built only from:', score_inputs)
print('Confirmed NOT used as score inputs:', forbidden.intersection(score_inputs) or 'none — clean')
print('Data restricted to month=2026-03 (mid-panel), never fact_content_daily_performance_sample',
      '(the sealed final month) per the panel warning in flyrank-data.')


Score built only from: {'query_total_impressions', 'competing_content_count'}
Confirmed NOT used as score inputs: none — clean
Data restricted to month=2026-03 (mid-panel), never fact_content_daily_performance_sample (the sealed final month) per the panel warning in flyrank-data.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.